# Group Relative Policy Optimization (GRPO) with veRL on Amazon SageMaker Training jobs
## Lab 4: LLM evaluation on Amazon SageMaker AI
In this lab, we are going to measure what GRPO actually bought us. Because GSM8K answers are checkable, we can report accuracy directly rather than relying on a similarity score or a judge model.

The other options in this solution reach for `BERTScore`, `ROUGE`, and LLM-as-a-judge, and they have to: when a model is trained to imitate a reference completion, "how good is this output" has no exact answer, so it gets approximated.

GRPO does not have that problem here. The reward function that trained the model is a rule -- extract the number after `####`, compare it to the ground truth -- so the same rule is a complete evaluation. A judge would add noise, not information.

We do this twice, from two independent directions:

1. **What veRL measured during training.** It validated on held-out data before the first step and again after the last one, so the before-and-after comparison is already recorded.
2. **What the deployed endpoint answers now.** Re-scoring the served model confirms that what Lab 3 deployed is the model training produced -- that the export, the merge, and the tokenizer fix all preserved it.

## Prerequisites

### Install requirements

In [ ]:
%pip install -r ./requirements.txt --upgrade

### Setup and dependencies

In [ ]:
import json
import os
import re

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()

sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sm_client = boto3.client("sagemaker")
logs_client = boto3.client("logs")
sagemaker_runtime = boto3.client("sagemaker-runtime")

sess = Session(default_bucket=sagemaker_session_bucket)

bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name

print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {region}")

Names re-derived, the same way Lab 3 built them.

In [ ]:
model_id = "Qwen/Qwen3-4B"
job_prefix = "train-qwen3-4b-grpo"

model_name = f"{model_id.split('/')[-1].replace('.', '-')}-grpo"
endpoint_name = f"{model_name}-endpoint"
ic_name = f"{model_name}-ic"

eval_dataset = "./tmp/gsm8k_eval.jsonl"

print(f"endpoint:  {endpoint_name}")
print(f"component: {ic_name}")

***

## What veRL measured during training
veRL writes one metrics line per step to the job's log stream, and on the steps where it validated, that line carries the accuracy on the held-out split. `val-core/openai/gsm8k/acc/mean@1` is the fraction of validation questions answered correctly.

The `@1` matters. During training each prompt gets `rollout_n` completions sampled at temperature, because GRPO needs a spread to compare within a group. Validation takes one completion per question, which is what you would actually serve, so this number is a fair estimate of deployed behaviour rather than a best-of-8.

In [ ]:
def get_last_job_name(job_name_prefix):
    results = sm_client.search(
        Resource="TrainingJob",
        SearchExpression={
            "Filters": [
                {"Name": "TrainingJobName", "Operator": "Contains", "Value": job_name_prefix},
                {"Name": "TrainingJobStatus", "Operator": "Equals", "Value": "Completed"},
            ]
        },
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=1,
    )["Results"]
    if not results:
        raise ValueError(f"no completed training job found with prefix '{job_name_prefix}'")
    return results[0]["TrainingJob"]["TrainingJobName"]


def fetch_validation_accuracy(job_name, metric="val-core/openai/gsm8k/acc/mean@1"):
    """Read per-step validation accuracy out of the training job's logs.

    veRL emits a single line per step of the form `step:N - key:value - key:value`,
    so the step and the metric are parsed off the same line. Steps that did not
    validate carry no validation keys and are skipped.

    Reads with GetLogEvents and filters here, rather than having FilterLogEvents
    filter server-side. Server-side would be the natural choice, but
    `logs:FilterLogEvents` is not granted by AmazonSageMakerFullAccess, so a standard
    execution role cannot call it at all: it fails with AccessDeniedException.
    `logs:GetLogEvents` is granted.
    """
    log_group = "/aws/sagemaker/TrainingJobs"
    streams = logs_client.describe_log_streams(
        logGroupName=log_group,
        logStreamNamePrefix=job_name,
    )["logStreams"]
    if not streams:
        raise ValueError(f"no log streams found for {job_name}")

    pattern = re.compile(rf"step:(\d+).*?{re.escape(metric)}:([0-9.]+)")
    found = {}

    for stream in streams:
        token = None
        while True:
            kwargs = {
                "logGroupName": log_group,
                "logStreamName": stream["logStreamName"],
                "startFromHead": True,
            }
            if token:
                kwargs["nextToken"] = token
            page = logs_client.get_log_events(**kwargs)
            for event in page["events"]:
                match = pattern.search(event["message"])
                if match:
                    found[int(match.group(1))] = float(match.group(2))
            # GetLogEvents marks the end by returning no events and echoing back the
            # same forward token, so both checks are needed to stop.
            next_token = page.get("nextForwardToken")
            if not page["events"] or next_token == token:
                break
            token = next_token

    if not found:
        raise ValueError(
            f"no {metric} values found in the logs for {job_name}. "
            "If test_freq was left at veRL's default of -1, validation never ran."
        )
    if len(found) == 1:
        only_step = next(iter(found))
        raise ValueError(
            f"only one validation point (step {only_step}) for {job_name}, so there is "
            "no baseline to compare against.\n\n"
            "The usual cause is a job that resumed from an earlier run's checkpoint. "
            "veRL's resume_mode is 'auto', so any checkpoint synced into "
            "/opt/ml/checkpoints is picked up, training restarts from that step, and "
            "the val_before_train pass never runs. Search the job log for "
            "'Load from checkpoint folder' to confirm. Lab 2 scopes its checkpoint "
            "prefix per run so this does not happen; a job submitted before that "
            "change can still show it, in which case rerun Lab 2."
        )
    return dict(sorted(found.items()))


job_name = get_last_job_name(job_prefix)
accuracy_by_step = fetch_validation_accuracy(job_name)

print(f"job: {job_name}\n")
for step, accuracy in accuracy_by_step.items():
    print(f"  step {step:>3}   {accuracy:.2%}")

The before-and-after, which is the headline result of the lab.

In [ ]:
steps = list(accuracy_by_step)
baseline_step, final_step = steps[0], steps[-1]
baseline, final = accuracy_by_step[baseline_step], accuracy_by_step[final_step]

print(f"baseline (step {baseline_step}): {baseline:.2%}")
print(f"trained  (step {final_step}): {final:.2%}")
print(f"change:              {final - baseline:+.2%}")

Plotted, with the baseline as a reference line. With `test_freq` set to 10 there are only two points; lowering it in Lab 2 validates more often and turns this into a curve, at the cost of a validation pass per interval.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))

labels = [f"step {s}" for s in steps]
values = [accuracy_by_step[s] * 100 for s in steps]

if len(steps) > 2:
    ax.plot(labels, values, marker="o", color="#232f3e")
else:
    ax.bar(labels, values, color=["#879596", "#ff9900"], width=0.5)

ax.axhline(baseline * 100, linestyle="--", linewidth=1, color="#879596")
ax.set_ylabel("GSM8K accuracy (%)")
ax.set_title(f"Validation accuracy over GRPO training\n{job_name}")
ax.set_ylim(0, 100)

for label, value in zip(labels, values):
    ax.annotate(f"{value:.1f}%", (label, value), textcoords="offset points",
                xytext=(0, 6), ha="center")

plt.tight_layout()
plt.show()

***

## Evaluate the deployed endpoint
The number above is what the trainer measured on its own weights, in memory. What Lab 3 deployed went through an FSDP-to-Hugging-Face merge, an S3 round trip, a tokenizer metadata rewrite, and a different inference engine. Re-scoring the endpoint is what confirms none of that changed the model.

We score with the same rule the reward function used, on the questions Lab 1 set aside.

In [ ]:
import pandas as pd

if not os.path.exists(eval_dataset):
    raise FileNotFoundError(
        f"{eval_dataset} not found. Run Lab 1, which writes the validation split there."
    )

eval_df = pd.read_json(eval_dataset, lines=True)

# How many questions to score. The full split is more accurate and takes longer; 100 is
# enough to see the difference and keeps the endpoint busy for a few minutes.
SAMPLE_SIZE = 100

sample = eval_df.head(SAMPLE_SIZE)
print(f"{len(eval_df)} questions available, scoring {len(sample)}")
sample.head(3)

Utility functions to ask the endpoint one question and to score the answer. The scoring rule is deliberately the same one veRL rewarded: take the number after the last `####`, strip commas, compare as text.

`max_tokens` is 1024 to match `max_response_length` from Lab 2, and that is not a detail to trim. This model reasons at length before it answers, and the `####` line is the *last* thing it writes -- so a completion cut short scores zero however correct its arithmetic was. Setting this to 600 made the endpoint appear to score 43% when it was really answering far better than that: 54 of 100 completions were truncated mid-sentence, several of them having already reached the right number. Evaluate with the budget the model was trained for.

In [ ]:
INSTRUCTION = 'Let\'s think step by step and output the final answer after "####".'


def extract_answer(text: str) -> str | None:
    """Take the final #### answer out of a completion."""
    matches = re.findall(r"####\s*(\-?[0-9.,]+)", text)
    if not matches:
        return None
    return matches[-1].replace(",", "").rstrip(".")


def ask(question: str, max_tokens: int = 1024) -> str:
    body = {
        "messages": [{"role": "user", "content": f"{question} {INSTRUCTION}"}],
        "max_tokens": max_tokens,
        # Greedy, so the score is reproducible.
        "temperature": 0.0,
    }
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        InferenceComponentName=ic_name,
        Body=json.dumps(body),
        ContentType="application/json",
    )
    payload = json.loads(response["Body"].read())
    return payload["choices"][0]["message"]["content"]


def score(row) -> dict:
    question = row["extra_info"]["question"]
    truth = row["reward_model"]["ground_truth"]
    try:
        completion = ask(question)
    except Exception as exc:  # keep one bad request from ending the run
        return {"question": question, "truth": truth, "predicted": None,
                "correct": False, "error": str(exc), "completion": ""}
    predicted = extract_answer(completion)
    return {
        "question": question,
        "truth": truth,
        "predicted": predicted,
        "correct": predicted == truth,
        "error": None,
        "completion": completion,
    }

Run the questions concurrently. Lab 3 set `SM_VLLM_MAX_NUM_SEQS` to 16, so vLLM batches up to 16 at once; eight workers keeps it busy without queueing much.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

started = time.time()

with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(score, (row for _, row in sample.iterrows())))

elapsed = time.time() - started
results_df = pd.DataFrame(results)

errors = int(results_df["error"].notna().sum())
endpoint_accuracy = float(results_df["correct"].mean())

print(f"scored {len(results_df)} questions in {elapsed:.0f}s")
if errors:
    print(f"  {errors} request(s) failed and were counted incorrect")
print(f"endpoint accuracy: {endpoint_accuracy:.2%}")

### Does the endpoint agree with training?
These two numbers are measured on different sample sizes and by different engines, so they will not match to the decimal. What matters is that the endpoint lands near the trained figure and nowhere near the baseline. A served model that scores like the baseline means the deployment is serving the original weights, not the trained ones.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

names = ["Base model\n(veRL, step 0)", f"GRPO trained\n(veRL, step {final_step})",
         "GRPO trained\n(endpoint)"]
values = [baseline * 100, final * 100, endpoint_accuracy * 100]
colors = ["#879596", "#ff9900", "#146eb4"]

bars = ax.bar(names, values, color=colors, width=0.55)
ax.set_ylabel("GSM8K accuracy (%)")
ax.set_title("GRPO: before, after, and as deployed")
ax.set_ylim(0, 100)

for bar, value in zip(bars, values):
    ax.annotate(f"{value:.1f}%", (bar.get_x() + bar.get_width() / 2, value),
                textcoords="offset points", xytext=(0, 6), ha="center")

plt.tight_layout()
plt.show()

print(f"trained vs baseline: {final - baseline:+.2%}")
print(f"endpoint vs trained: {endpoint_accuracy - final:+.2%}")

### Look at what it actually wrote
An accuracy number hides how the answer was reached. GRPO rewarded only the final number, so the reasoning in between was never scored directly -- it improved because reasoning that reaches correct answers gets reinforced.

In [ ]:
correct_rows = results_df[results_df["correct"]]
wrong_rows = results_df[~results_df["correct"] & results_df["error"].isna()]

for label, frame in (("CORRECT", correct_rows), ("INCORRECT", wrong_rows)):
    if frame.empty:
        continue
    row = frame.iloc[0]
    print(f"===== {label} =====")
    print(f"Q: {row['question']}")
    print(f"expected {row['truth']}, model answered {row['predicted']}\n")
    print(row["completion"][:1200])
    print()

It is worth checking *why* the wrong ones were wrong, because there are two different failures hiding in one number: the model did the arithmetic wrong, or it never produced a parseable `####` line at all. The second kind scores zero during training too, so a high rate of it means the reward signal was being wasted.

In [ ]:
unparseable = int(results_df["predicted"].isna().sum())
wrong_answer = int((~results_df["correct"] & results_df["predicted"].notna()).sum())

print(f"correct:               {int(results_df['correct'].sum()):>4}")
print(f"wrong answer:          {wrong_answer:>4}")
print(f"no #### answer found:  {unparseable:>4}")

if unparseable:
    print("\nA missing #### answer scores zero however good the reasoning was.")
    print("Raising max_tokens is the first thing to try: a truncated completion")
    print("never reaches its final line.")

Save the results into files.

In [ ]:
os.makedirs("./tmp/evaluation_results", exist_ok=True)

results_df.drop(columns=["completion"]).to_csv(
    "./tmp/evaluation_results/grpo_endpoint_eval.csv", index=False
)

summary = {
    "job_name": job_name,
    "baseline_accuracy": baseline,
    "trained_accuracy": final,
    "endpoint_accuracy": endpoint_accuracy,
    "questions_scored": len(results_df),
    "accuracy_by_step": accuracy_by_step,
}

with open("./tmp/evaluation_results/grpo_summary.json", "w") as handle:
    json.dump(summary, handle, indent=2)

print(json.dumps(summary, indent=2))

***

### Delete resources
The endpoint is still running and still billing. Run these unless you are going on to query it further -- and if you are, come back and run them when you are done.

In [ ]:
from sagemaker.core.resources import InferenceComponent

# Delete inference component
InferenceComponent.get(inference_component_name=ic_name).delete()

In [ ]:
from sagemaker.core.resources import Endpoint

# Delete endpoint -- this is the one that stops the hourly charge
Endpoint.get(endpoint_name=endpoint_name).delete()

In [ ]:
from sagemaker.core.resources import Model

# Delete model
Model.get(model_name=model_name).delete()

In [ ]:
from sagemaker.core.resources import EndpointConfig

# Delete endpoint config
EndpointConfig.get(endpoint_config_name=model_name).delete()